<a href="https://colab.research.google.com/github/Venky1322/Vision-Transformers-with-Temporal-Modelling-for-Video-Violence-Detection/blob/main/Violence_Detection_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Violence Detection using Vision Transformer (ViT)

## Dataset Limited to 600 Videos (300 Violence + 300 Non-Violence)

This notebook implements a complete pipeline for violence detection using Vision Transformer on the Real Life Violence Situations Dataset from Kaggle.

**Features:**
- Uses pretrained Vision Transformer (ViT) model
- Frame-based video classification
- Data augmentation for improved performance
- Comprehensive evaluation metrics
- Video-level prediction with confidence scores

**Dataset Configuration:**
- Total Videos: 600 (300 Violence + 300 Non-Violence)
- Frames per Video: 30 frames
- Train/Val Split: 80/20
- Estimated Dataset Size: ~2-3 GB
- Estimated Training Time: ~15-20 minutes

**Note:** Enable GPU in Runtime → Change runtime type → GPU (T4 recommended)

## 1. Install Required Packages

In [ ]:
!pip install -q transformers torch albumentations opencv-python scikit-learn tqdm matplotlib
!pip install -q kaggle

## 2. Setup Kaggle API

**Important Steps:**
1. Go to [Kaggle](https://www.kaggle.com/) → Account → API → Create New API Token
2. Download the `kaggle.json` file
3. Upload it in the cell below

In [ ]:
# Upload kaggle.json file
from google.colab import files
print('Please upload your kaggle.json file:')
uploaded = files.upload()

In [ ]:
# Setup Kaggle credentials
import os

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print('✓ Kaggle API configured successfully!')

## 3. Download and Extract Dataset (Limited to 600 Videos)

In [ ]:
# Download the Real Life Violence Situations Dataset
print("Downloading dataset from Kaggle...")
!kaggle datasets download -d mohamedmustafa/real-life-violence-situations-dataset

print("\nExtracting dataset...")
# First, check what's in the zip file
!unzip -l real-life-violence-situations-dataset.zip | head -20

# Extract the dataset
!unzip -q real-life-violence-situations-dataset.zip -d /content/temp_extract

# Check the structure
print("\n🔍 Checking extracted structure...")
!ls -la /content/temp_extract/

# The dataset might be in a subdirectory, so let's find the actual structure
import os
import shutil

temp_path = '/content/temp_extract'
dataset_path = '/content/dataset'

# Remove old dataset folder if exists
if os.path.exists(dataset_path):
    shutil.rmtree(dataset_path)

# Find where Violence and NonViolence folders are
violence_found = None
nonviolence_found = None

for root, dirs, files in os.walk(temp_path):
    if 'Violence' in dirs:
        violence_found = os.path.join(root, 'Violence')
    if 'NonViolence' in dirs:
        nonviolence_found = os.path.join(root, 'NonViolence')

if violence_found and nonviolence_found:
    # Create the dataset directory structure
    os.makedirs(dataset_path, exist_ok=True)

    # Move or copy the folders
    parent_dir = os.path.dirname(violence_found)
    shutil.move(violence_found, os.path.join(dataset_path, 'Violence'))
    shutil.move(nonviolence_found, os.path.join(dataset_path, 'NonViolence'))

    print(f"✓ Dataset organized successfully!")
else:
    print("⚠️ Could not find Violence/NonViolence folders in expected structure")
    print("Checking alternative structures...")

    # List everything in temp extract
    for item in os.listdir(temp_path):
        print(f"  - {item}")

# Clean up temp directory
if os.path.exists(temp_path):
    shutil.rmtree(temp_path)

# Verify final structure
print("\n✓ Final dataset structure:")
!ls -la /content/dataset/

print("\nVideos in Violence folder (total):")
violence_count = !ls /content/dataset/Violence/ | wc -l
print(f"Total violence videos available: {violence_count[0]}")

print("\nVideos in NonViolence folder (total):")
nonviolence_count = !ls /content/dataset/NonViolence/ | wc -l
print(f"Total non-violence videos available: {nonviolence_count[0]}")


## 4. Import Libraries and Configure Parameters

In [ ]:
import os
import random
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from transformers import ViTImageProcessor, ViTForImageClassification, ViTConfig
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print('✓ All libraries imported successfully!')

In [ ]:
# Configuration Parameters - LIMITED TO 600 VIDEOS (300 PER CLASS)
DATASET_PATHS = {
    'violence': '/content/dataset/Violence',
    'nonviolence': '/content/dataset/NonViolence'
}

# KEY SETTING: Limit videos to 300 per class (600 total)
MAX_VIDEOS_PER_CLASS = 300  # ← 300 violence + 300 non-violence = 600 total

MODEL_NAME = 'google/vit-base-patch16-224-in21k'
OUTPUT_MODEL_PATH = '/content/vit_violence_detection_600videos.pth'
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 10
LR = 2e-5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_FRAMES_PER_VIDEO = 30
FRAME_SAMPLE_STRATEGY = 'uniform'
THRESHOLD = 0.5

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('='*60)
print('CONFIGURATION SUMMARY')
print('='*60)
print(f'  Videos per Class: {MAX_VIDEOS_PER_CLASS}')
print(f'  Total Videos: {MAX_VIDEOS_PER_CLASS * 2}')
print(f'  Device: {DEVICE}')
print(f'  Batch Size: {BATCH_SIZE}')
print(f'  Epochs: {EPOCHS}')
print(f'  Learning Rate: {LR}')
print(f'  Frames per Video: {MAX_FRAMES_PER_VIDEO}')
print(f'  Total Dataset Size (approx): ~2-3 GB')
print(f'  Est. Training Time: ~15-20 minutes')
print('='*60)

## 5. Verify Dataset Structure

In [ ]:
# Diagnostic: Check dataset extraction and count available videos
print("🔍 DATASET DIAGNOSTIC CHECK")
print("="*70)

import os
import cv2
from pathlib import Path

# Check if dataset directory exists
dataset_root = '/content/dataset'
if os.path.exists(dataset_root):
    print(f"✓ Dataset directory exists: {dataset_root}")

    # List all contents
    print(f"\nContents of {dataset_root}:")
    items = os.listdir(dataset_root)
    for item in items:
        item_path = os.path.join(dataset_root, item)
        if os.path.isdir(item_path):
            num_files = len(os.listdir(item_path))
            print(f"  📁 {item}/ ({num_files} items)")

            # Show first few files in each directory
            if num_files > 0:
                files = sorted(os.listdir(item_path))[:3]
                for f in files:
                    print(f"      ├─ {f}")
                if num_files > 3:
                    print(f"      ├─ ... and {num_files - 3} more files")
        else:
            print(f"  📄 {item}")

    # Calculate video statistics
    print("\n" + "="*70)
    print("VIDEO STATISTICS")
    print("="*70)

    violence_path = os.path.join(dataset_root, 'Violence')
    nonviolence_path = os.path.join(dataset_root, 'NonViolence')

    if os.path.exists(violence_path):
        violence_videos = len(os.listdir(violence_path))
        print(f"Violence videos available: {violence_videos}")

    if os.path.exists(nonviolence_path):
        nonviolence_videos = len(os.listdir(nonviolence_path))
        print(f"Non-violence videos available: {nonviolence_videos}")

    print(f"\nLimit per class: {MAX_VIDEOS_PER_CLASS}")
    print(f"Total videos to use: {MAX_VIDEOS_PER_CLASS * 2}")

else:
    print(f"✗ Dataset directory NOT found: {dataset_root}")
    print("\nPlease run the dataset download cell (Section 3) first")

print("="*70)


## 6. Define Data Augmentation

In [ ]:
# Training augmentation (with random transformations)
train_transform = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.RandomBrightnessContrast(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.GaussNoise(p=0.3),
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.8, 1.0), p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# Validation augmentation (no random transformations)
val_transform = A.Compose([
    A.Resize(height=IMG_SIZE, width=IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

print('✓ Data augmentation pipelines created')

## 7. Create Custom Dataset Class

In [ ]:
class FrameDataset(Dataset):
    """
    Custom Dataset for video frame extraction and classification.
    """
    def __init__(self, video_paths, labels, transform, max_frames_per_video=30, strategy='uniform'):
        self.video_paths = video_paths
        self.labels = labels
        self.transform = transform
        self.max_frames = max_frames_per_video
        self.strategy = strategy

        # Pre-extract frames from all videos
        self.frames = []
        self.frame_labels = []

        print(f'Extracting frames from {len(video_paths)} videos...')
        for video_path, label in tqdm(zip(video_paths, labels), total=len(video_paths)):
            extracted_frames = self._extract_frames(video_path)
            self.frames.extend(extracted_frames)
            self.frame_labels.extend([label] * len(extracted_frames))

        print(f'Total frames extracted: {len(self.frames)}')

    def _extract_frames(self, video_path):
        """Extract frames from a video file."""
        frames = []
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            return frames

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames == 0:
            cap.release()
            return frames

        # Select frame indices
        if self.strategy == 'uniform':
            frame_indices = np.linspace(0, total_frames - 1, min(self.max_frames, total_frames), dtype=int)
        else:
            frame_indices = sorted(random.sample(range(total_frames), min(self.max_frames, total_frames)))

        # Extract frames
        for idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ret, frame = cap.read()
            if ret:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)

        cap.release()
        return frames

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, idx):
        frame = self.frames[idx]
        label = self.frame_labels[idx]

        if self.transform:
            transformed = self.transform(image=frame)
            frame = transformed['image']

        return frame, label

print('✓ FrameDataset class defined')

## 8. Load and Prepare Dataset (LIMITED TO 600 VIDEOS)

In [ ]:
def load_video_paths(dataset_paths, max_videos_per_class=50):
    """
    Load video file paths from the dataset directories.
    Limited to max_videos_per_class per class.
    """
    video_paths = []
    labels = []

    # Check if paths exist
    print("Checking dataset paths...")
    for key, path in dataset_paths.items():
        if os.path.exists(path):
            print(f"✓ {key} path exists: {path}")
        else:
            print(f"✗ {key} path NOT found: {path}")

    # Load violence videos (label = 1)
    violence_dir = Path(dataset_paths['violence'])

    # Try multiple video extensions
    violence_videos = []
    for ext in ['*.mp4', '*.avi', '*.mov', '*.MP4', '*.AVI', '*.MOV']:
        violence_videos.extend(list(violence_dir.glob(ext)))

    # LIMIT TO 300 VIDEOS
    if len(violence_videos) > max_videos_per_class:
        print(f"Found {len(violence_videos)} violence videos, limiting to {max_videos_per_class}")
        violence_videos = sorted(violence_videos)[:max_videos_per_class]

    video_paths.extend([str(v) for v in violence_videos])
    labels.extend([1] * len(violence_videos))

    # Load non-violence videos (label = 0)
    nonviolence_dir = Path(dataset_paths['nonviolence'])

    nonviolence_videos = []
    for ext in ['*.mp4', '*.avi', '*.mov', '*.MP4', '*.AVI', '*.MOV']:
        nonviolence_videos.extend(list(nonviolence_dir.glob(ext)))

    # LIMIT TO 300 VIDEOS
    if len(nonviolence_videos) > max_videos_per_class:
        print(f"Found {len(nonviolence_videos)} non-violence videos, limiting to {max_videos_per_class}")
        nonviolence_videos = sorted(nonviolence_videos)[:max_videos_per_class]

    video_paths.extend([str(v) for v in nonviolence_videos])
    labels.extend([0] * len(nonviolence_videos))

    print(f'\n' + '='*60)
    print('DATASET SUMMARY')
    print('='*60)
    print(f'Loaded {len(violence_videos)} violence videos')
    print(f'Loaded {len(nonviolence_videos)} non-violence videos')
    print(f'Total videos: {len(video_paths)}')
    print(f'Total frames (approx): {len(video_paths) * MAX_FRAMES_PER_VIDEO}')
    print('='*60)

    # Error checking
    if len(video_paths) == 0:
        print("\n" + "="*70)
        print("ERROR: No videos found!")
        print("="*70)
        print("\nPossible solutions:")
        print("1. Check if the dataset was extracted correctly")
        print("2. Verify the folder structure after unzipping")
        print("3. Run the diagnostic cell above to see actual contents")
        print("="*70)

        # Try to list what's actually in the dataset directory
        print("\nActual directory contents:")
        dataset_root = '/content/dataset'
        if os.path.exists(dataset_root):
            print(f"\nContents of {dataset_root}:")
            for item in os.listdir(dataset_root):
                item_path = os.path.join(dataset_root, item)
                if os.path.isdir(item_path):
                    print(f"  📁 {item}/ ({len(os.listdir(item_path))} items)")
                else:
                    print(f"  📄 {item}")
        else:
            print(f"Directory {dataset_root} does not exist!")

    return video_paths, labels

# Load video paths with 300 per class limit
video_paths, labels = load_video_paths(DATASET_PATHS, MAX_VIDEOS_PER_CLASS)

# Only proceed with split if we have videos
if len(video_paths) > 0:
    # Split into train and validation sets (80-20 split)
    train_videos, val_videos, train_labels, val_labels = train_test_split(
        video_paths, labels, test_size=0.2, random_state=SEED, stratify=labels
    )

    print(f'\nTrain/Val Split:')
    print(f'  Train videos: {len(train_videos)}')
    print(f'  Validation videos: {len(val_videos)}')
else:
    print("\n⚠️ Cannot proceed without video files. Please check the dataset extraction.")
    raise ValueError("No videos found in the dataset directories.")


## 9. Create DataLoaders

In [ ]:

# Create datasets
print('Creating training dataset...')
train_dataset = FrameDataset(
    train_videos,
    train_labels,
    train_transform,
    MAX_FRAMES_PER_VIDEO,
    FRAME_SAMPLE_STRATEGY
)

print('\nCreating validation dataset...')
val_dataset = FrameDataset(
    val_videos,
    val_labels,
    val_transform,
    MAX_FRAMES_PER_VIDEO,
    FRAME_SAMPLE_STRATEGY
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f'\n' + '='*60)
print('DATALOADER SUMMARY')
print('='*60)
print(f'Train batches: {len(train_loader)}')
print(f'Validation batches: {len(val_loader)}')
print(f'Train frames: {len(train_dataset)}')
print(f'Validation frames: {len(val_dataset)}')
print('='*60)

## 10. Load Vision Transformer Model

In [ ]:
# Load pretrained ViT model
print('Loading Vision Transformer model...')

config = ViTConfig.from_pretrained(MODEL_NAME, num_labels=2)
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    config=config,
    ignore_mismatched_sizes=True
)
model.to(DEVICE)

# Setup optimizer and loss
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\n' + '='*60)
print('MODEL INFORMATION')
print('='*60)
print(f'Model: {MODEL_NAME}')
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Device: {DEVICE}')
print('='*60)

## 11. Define Training and Evaluation Functions

In [ ]:
def train_epoch(model, train_loader, optimizer, criterion, device):
    """Train the model for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc='Training')
    for frames, labels in pbar:
        frames = frames.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(frames)
        logits = outputs.logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(logits, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        pbar.set_postfix({
            'loss': running_loss / (pbar.n + 1),
            'acc': 100. * correct / total
        })

    return running_loss / len(train_loader), 100. * correct / total


def evaluate(model, val_loader, criterion, device):
    """Evaluate the model on validation set."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validation')
        for frames, labels in pbar:
            frames = frames.to(device)
            labels = labels.to(device)

            outputs = model(frames)
            logits = outputs.logits
            loss = criterion(logits, labels)

            running_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            pbar.set_postfix({
                'loss': running_loss / (pbar.n + 1),
                'acc': 100. * correct / total
            })

    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100. * correct / total
    f1 = f1_score(all_labels, all_preds, average='binary')

    return epoch_loss, epoch_acc, f1, all_preds, all_labels

print('✓ Training and evaluation functions defined')

## 12. Train the Model

In [ ]:
# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'val_f1': []
}

best_val_acc = 0.0
best_f1 = 0.0

print(f'\n' + '='*60)
print(f'STARTING TRAINING FOR {EPOCHS} EPOCHS')
print('='*60 + '\n')

for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    print('-' * 60)

    # Train
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)

    # Validate
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, criterion, DEVICE)

    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    # Print summary
    print(f'\nEpoch {epoch+1} Summary:')
    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
    print(f'  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | Val F1: {val_f1:.4f}')

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_f1 = val_f1
        torch.save(model.state_dict(), OUTPUT_MODEL_PATH)
        print(f'  ✓ Best model saved! (Val Acc: {val_acc:.2f}%)')

print('\n' + '='*60)
print('TRAINING COMPLETED!')
print('='*60)
print(f'Best Validation Accuracy: {best_val_acc:.2f}%')
print(f'Best F1 Score: {best_f1:.4f}')
print('='*60)

## 13. Visualize Training Results

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss plot
axes[0].plot(history['train_loss'], label='Train Loss', marker='o', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', marker='s', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy plot
axes[1].plot(history['train_acc'], label='Train Acc', marker='o', linewidth=2)
axes[1].plot(history['val_acc'], label='Val Acc', marker='s', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# F1 Score plot
axes[2].plot(history['val_f1'], label='Val F1', marker='d', linewidth=2, color='green')
axes[2].set_xlabel('Epoch', fontsize=12)
axes[2].set_ylabel('F1 Score', fontsize=12)
axes[2].set_title('Validation F1 Score', fontsize=14, fontweight='bold')
axes[2].legend(fontsize=11)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print('✓ Training history plots saved to /content/training_history.png')

## 14. Final Model Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load(OUTPUT_MODEL_PATH))
model.eval()

# Get predictions
_, val_acc, val_f1, predictions, true_labels = evaluate(model, val_loader, criterion, DEVICE)

print('\n' + '='*60)
print('FINAL MODEL EVALUATION')
print('='*60)
print(f'Validation Accuracy: {val_acc:.2f}%')
print(f'F1 Score: {val_f1:.4f}')
print('\nDetailed Classification Report:')
print(classification_report(true_labels, predictions,
                           target_names=['Non-Violence', 'Violence']))

# Confusion Matrix
cm = confusion_matrix(true_labels, predictions)
print('\nConfusion Matrix:')
print(cm)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, cmap='Blues')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Non-Violence', 'Violence'], fontsize=12)
ax.set_yticklabels(['Non-Violence', 'Violence'], fontsize=12)
ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax.set_ylabel('True', fontsize=12, fontweight='bold')
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')

for i in range(2):
    for j in range(2):
        text = ax.text(j, i, cm[i, j], ha='center', va='center',
                      color='white' if cm[i, j] > cm.max()/2 else 'black',
                      fontsize=16, fontweight='bold')

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Confusion matrix saved to /content/confusion_matrix.png')

## 15. Video-Level Prediction Function

In [ ]:
def predict_video(video_path, model, transform, device, max_frames=30, threshold=0.5):
    """
    Predict violence in a video using frame-level predictions.
    Returns: (prediction, confidence)
    """
    model.eval()

    # Extract frames
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        return None, None

    # Sample frames uniformly
    frame_indices = np.linspace(0, total_frames - 1, min(max_frames, total_frames), dtype=int)

    frames = []
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

    cap.release()

    if not frames:
        return None, None

    # Process frames
    frame_tensors = []
    for frame in frames:
        transformed = transform(image=frame)
        frame_tensor = transformed['image']
        frame_tensors.append(frame_tensor)

    # Batch prediction
    frame_batch = torch.stack(frame_tensors).to(device)

    with torch.no_grad():
        outputs = model(frame_batch)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        violence_probs = probs[:, 1].cpu().numpy()

    # Video-level prediction (average)
    avg_violence_prob = np.mean(violence_probs)
    video_prediction = 1 if avg_violence_prob >= threshold else 0

    return video_prediction, avg_violence_prob

print('✓ Video prediction function defined')

## 16. Test on Sample Validation Videos

In [ ]:
# Test on sample validation videos
print('Testing on sample validation videos...\n')

num_samples = 10
sample_indices = random.sample(range(len(val_videos)), min(num_samples, len(val_videos)))

correct_predictions = 0

print('='*70)
for idx in sample_indices:
    video_path = val_videos[idx]
    true_label = val_labels[idx]

    prediction, confidence = predict_video(
        video_path, model, val_transform, DEVICE, MAX_FRAMES_PER_VIDEO, THRESHOLD
    )

    if prediction is not None:
        video_name = Path(video_path).name
        true_class = 'Violence' if true_label == 1 else 'Non-Violence'
        pred_class = 'Violence' if prediction == 1 else 'Non-Violence'
        status = '✓' if prediction == true_label else '✗'

        if prediction == true_label:
            correct_predictions += 1

        print(f'{status} Video: {video_name[:35]}')
        print(f'   True: {true_class:15s} | Pred: {pred_class:15s} | Conf: {confidence:.2%}')

print('='*70)
print(f'\nSample Accuracy: {correct_predictions}/{len(sample_indices)} ({100*correct_predictions/len(sample_indices):.1f}%)')

## 17. Upload Video

In [ ]:
# Upload custom video
from google.colab import files

print('Upload a video file to test:')
print('(Supported formats: .mp4, .avi, .mov)\n')
uploaded = files.upload()

if uploaded:
    video_file = list(uploaded.keys())[0]

    print(f'\nAnalyzing video: {video_file}...')
    prediction, confidence = predict_video(
        video_file, model, val_transform, DEVICE, MAX_FRAMES_PER_VIDEO, THRESHOLD
    )

    if prediction is not None:
        pred_class = 'Violence' if prediction == 1 else 'Non-Violence'

        print('\n' + '='*70)
        print('PREDICTION RESULT')
        print('='*70)
        print(f'Video: {video_file}')
        print(f'Prediction: {pred_class}')
        print(f'Confidence: {confidence:.2%}')
        print('='*70)
    else:
        print('Error: Could not process the video file.')
else:
    print('No file uploaded.')